# LeNet-5 Training & Comparison on CIFAR-10

This notebook demonstrates the training of the classic **LeNet-5** architecture on the **CIFAR-10** dataset. We compare the mathematical correctness and optimization convergence of **Gradience** (running on CPU) against **PyTorch** (running on GPU/CUDA if available).

## 1. Imports and Setup

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

sys.path.append(os.path.abspath('..'))

from gradience.tensor import Tensor
from gradience.nn.models.lenet import LeNet5
from gradience.optim.sgd import SGD

## 2. PyTorch LeNet-5 Reference Model

In [ ]:
class PtLeNet5(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 6, kernel_size=5, stride=1, padding=0)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc3 = nn.Linear(16 * 5 * 5, 120)
        self.fc4 = nn.Linear(120, 84)
        self.fc5 = nn.Linear(84, num_classes)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.pool(self.tanh(self.conv1(x)))
        x = self.pool(self.tanh(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.tanh(self.fc3(x))
        x = self.tanh(self.fc4(x))
        x = self.fc5(x)
        return x

def copy_weights_lenet(pt_model, gr_model):
    convs = [('conv1', 'conv1'), ('conv2', 'conv2')]
    for pt_name, gr_name in convs:
        pt_conv = getattr(pt_model, pt_name)
        gr_conv = getattr(gr_model, gr_name)
        gr_conv.weight.data[...] = pt_conv.weight.cpu().detach().numpy()
        if gr_conv.bias is not None:
            gr_conv.bias.data[...] = pt_conv.bias.cpu().detach().numpy()

    linears = [('fc3', 'fc3'), ('fc4', 'fc4'), ('fc5', 'fc5')]
    for pt_name, gr_name in linears:
        pt_linear = getattr(pt_model, pt_name)
        gr_linear = getattr(gr_model, gr_name)
        gr_linear.weight.data[...] = pt_linear.weight.cpu().detach().numpy().T
        if gr_linear.bias is not None:
            gr_linear.bias.data[...] = pt_linear.bias.cpu().detach().numpy()

## 3. CIFAR-10 Dataset Loading & Processing

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

num_classes = 10
batch_size = 4
height, width = 32, 32

transform = transforms.Compose([
    transforms.Resize((height, width)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

X_list = []
y_list = []
for idx in range(32):
    img_tensor, label = cifar_train[idx]
    X_list.append(img_tensor.numpy())
    y_list.append(label)

X_data = np.stack(X_list, axis=0).astype(np.float64)
y_data = np.array(y_list)
y_data_oh = np.eye(num_classes)[y_data]

print(f"Dataset shape: {X_data.shape}")
print(f"Labels shape:  {y_data_oh.shape}")

## 4. Single Forward/Backward Pass Verification

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch Device: {device}")

gr_model = LeNet5(in_channels=3, num_classes=num_classes)
pt_model = PtLeNet5(in_channels=3, num_classes=num_classes).double().to(device)

copy_weights_lenet(pt_model, gr_model)

gr_model.eval()
pt_model.eval()

x_pt = torch.tensor(X_data[:batch_size], requires_grad=True, dtype=torch.float64, device=device)
x_gr = Tensor(X_data[:batch_size], requires_grad=True)

y_pt_oh = torch.tensor(y_data_oh[:batch_size], dtype=torch.float64, device=device)
y_gr_oh = Tensor(y_data_oh[:batch_size])

out_pt = pt_model(x_pt)
loss_pt = ((out_pt - y_pt_oh) ** 2).mean()
loss_pt.backward()

out_gr = gr_model(x_gr)
loss_gr = ((out_gr - y_gr_oh) ** 2).mean()
loss_gr.backward()

diff_logits = np.max(np.abs(out_gr.data - out_pt.cpu().detach().numpy()))
diff_loss = np.abs(loss_gr.item() - loss_pt.item())
diff_grad_in = np.max(np.abs(x_gr.grad - x_pt.grad.cpu().numpy()))

print(f"Logits Max Diff: {diff_logits:.2e}")
print(f"Loss Diff:        {diff_loss:.2e}")
print(f"Input Grad Diff:  {diff_grad_in:.2e}")

## 5. Mini Training Loop Convergence Comparison

In [ ]:
gr_model = LeNet5(in_channels=3, num_classes=num_classes)
pt_model = PtLeNet5(in_channels=3, num_classes=num_classes).double().to(device)

copy_weights_lenet(pt_model, gr_model)

gr_model.eval()
pt_model.eval()

lr = 0.01
gr_opt = SGD(gr_model.parameters(), lr=lr)
pt_opt = torch.optim.SGD(pt_model.parameters(), lr=lr)

gr_losses = []
pt_losses = []

for epoch in range(5):
    epoch_gr_loss = 0.0
    epoch_pt_loss = 0.0
    
    for i in range(0, len(X_data), batch_size):
        xb = X_data[i:i+batch_size]
        yb = y_data_oh[i:i+batch_size]
        
        xb_pt = torch.tensor(xb, dtype=torch.float64, device=device)
        yb_pt = torch.tensor(yb, dtype=torch.float64, device=device)
        pt_opt.zero_grad()
        out_pt_loop = pt_model(xb_pt)
        loss_pt_loop = ((out_pt_loop - yb_pt) ** 2).mean()
        loss_pt_loop.backward()
        pt_opt.step()
        epoch_pt_loss += loss_pt_loop.item()
        
        xb_gr = Tensor(xb)
        yb_gr = Tensor(yb)
        gr_opt.zero_grad()
        out_gr_loop = gr_model(xb_gr)
        loss_gr_loop = ((out_gr_loop - yb_gr) ** 2).mean()
        loss_gr_loop.backward()
        gr_opt.step()
        epoch_gr_loss += loss_gr_loop.item()
        
    gr_losses.append(epoch_gr_loss / 8.0)
    pt_losses.append(epoch_pt_loss / 8.0)
    print(f"Epoch {epoch+1} | Gradience Loss: {gr_losses[-1]:.6f} | PyTorch Loss: {pt_losses[-1]:.6f}")

## 6. Plot Loss Convergence

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gr_losses, label="Gradience (CPU)", marker="o", linewidth=2)
plt.plot(pt_losses, label=f"PyTorch ({device.type.upper()})", linestyle="--", marker="x", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error Loss")
plt.title("Convergence Comparison on CIFAR-10: Gradience vs. PyTorch")
plt.legend()
plt.grid(True)
plt.savefig("lenet_convergence.png")
plt.show()